In [47]:
!pip install PyMuPDF

import fitz

In [48]:
# Extracts PDF text with pages
def read_pdf_with_page_numbers(file_path):
  doc = fitz.open(file_path)
  data = []
  for page_num in range(len(doc)):
    page = doc[page_num]
    text = page.get_text("text")
    data.append({"page": page_num+1, "content": text})

  return data;

pages = read_pdf_with_page_numbers("/content/drive/MyDrive/JOB TASK DATASET/policy_file.pdf");
print(pages[0])

{'page': 1, 'content': '1.2 \nFINANCIAL POLICY OBJECTIVES AND STRATEGIES \n \nSTATEMENT \nThe presentation and preparation of the Territory’s Budget is provided for in sections 11 and \n11A of the Financial Management Act 1996 (the Act).   \nThe purpose of the financial policy objectives and strategies statement is to make transparent \nthe Government’s financial strategies and to establish a benchmark for evaluating the \nGovernment’s conduct of financial policy.  The statement is also consistent with section \n11(1)(a) of the Act. \nStrategic Priorities and Financial Policy \nIn this budget, the Government continues its commitment to the principles of responsible \nfinancial management. \nThe financial objectives and the key measures below outline the Government’s financial \npolicy.  Strategic priorities, as they relate to the Territory’s Budget, are summarised as: \n• maintain a balanced budget over the economic cycle; \n• maintain low levels of debt; \n• provide the highest possib

In [49]:
# Get specific text by Keywords
keywords = ["budget", "debt", "infrastructure"]

def extract_relevant_info(pages, keywords):
    results = []
    for page in pages:
        text = page["content"].lower()
        for kw in keywords:
            if kw in text:
                results.append({
                    "keyword": kw,
                    "page": page.get("page", page.get("section")),
                    "excerpt": page["content"][:300]
                })
    return results


important_info = extract_relevant_info(pages, keywords)
for info in important_info:
    print(info)


{'keyword': 'budget', 'page': 1, 'excerpt': '1.2 \nFINANCIAL POLICY OBJECTIVES AND STRATEGIES \n \nSTATEMENT \nThe presentation and preparation of the Territory’s Budget is provided for in sections 11 and \n11A of the Financial Management Act 1996 (the Act).   \nThe purpose of the financial policy objectives and strategies statement is to make trans'}
{'keyword': 'debt', 'page': 1, 'excerpt': '1.2 \nFINANCIAL POLICY OBJECTIVES AND STRATEGIES \n \nSTATEMENT \nThe presentation and preparation of the Territory’s Budget is provided for in sections 11 and \n11A of the Financial Management Act 1996 (the Act).   \nThe purpose of the financial policy objectives and strategies statement is to make trans'}
{'keyword': 'infrastructure', 'page': 1, 'excerpt': '1.2 \nFINANCIAL POLICY OBJECTIVES AND STRATEGIES \n \nSTATEMENT \nThe presentation and preparation of the Territory’s Budget is provided for in sections 11 and \n11A of the Financial Management Act 1996 (the Act).   \nThe purpose of the fina

In [50]:
!pip install transformers sentence-transformers faiss-cpu accelerate
!pip install faiss-cpu sentence-transformers
from transformers import pipeline
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

In [51]:
# Convert text to vector
data = important_info

model = SentenceTransformer("all-MiniLM-L6-v2")


texts = [item["excerpt"] for item in data]
embeddings = model.encode(texts, convert_to_numpy=True)

dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embeddings)

print(f"Stored {index.ntotal} policy chunks in FAISS")

Stored 15 policy chunks in FAISS


In [52]:
# Answer the Question
qa_model = pipeline("text2text-generation", model="google/flan-t5-base")

conversation_history = []

def search(query, top_k=3):
    query_vec = model.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_vec, top_k)
    results = [data[i] for i in indices[0]]
    return results

def chatbot_response(user_input):
    conversation_history.append({"role": "user", "content": user_input})

    results = search(user_input)
    context_text = "\n".join([f"Page {r['page']}: {r['excerpt']}" for r in results])

    prompt = f"""
Answer the question based on the following policy document excerpts:
{context_text}

Question: {user_input}
Answer:
    """

    reply = qa_model(prompt, max_new_tokens=256, do_sample=False)[0]["generated_text"]

    conversation_history.append({"role": "assistant", "content": reply})
    return reply

Device set to use cpu


In [53]:
# Sample example
print(chatbot_response("What is the budget?"))
print(chatbot_response("And what about debt?"))
print(chatbot_response("What about infrastructure?"))

The Budget supports community welfare and development. Investments are made with both short and long-term objectives in mind. Various projects associated with implementing the Shaping Our Territory plan and ACT Energy Wise Program are provided for, while the Asbestos Assessment Task Fo
This objective provides an indication of the Government’s ability to meet its debt obligations without impacting on services, and in the current case for the General Government Sector, a net interest return for the Territory. The Government’s target for this measure is to ensure that the level o
achieving and maintaining levels of Territory net worth to provide a buffer against factors that may impact adversely on levels of Territory net worth in the future; (d) managing prudently the fiscal risks of the Territory; (e) pursuing spending and taxing policies that are consistent wit
